# IPL EDA and Statistics
Exploratory Data Analysis and Statistics for IPL Matches (2008-2020).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency

matches = pd.read_csv('../data/matches.csv')
deliveries = pd.read_csv('../data/deliveries.csv')

## 1. Distribution of first innings totals by venue

In [ ]:
first_innings = deliveries[deliveries['inning'] == 1].groupby('match_id')['total_runs'].sum().reset_index()
first_innings = first_innings.merge(matches[['id', 'venue']], left_on='match_id', right_on='id')
plt.figure(figsize=(12, 6))
sns.histplot(data=first_innings, x='total_runs', bins=30, kde=True)
plt.title('Distribution of First Innings Totals')
plt.xlabel('Total Runs')
plt.ylabel('Frequency')
plt.show()

## 2. Run rate progression curves by innings

In [ ]:
overs = deliveries.groupby(['match_id', 'inning', 'over'])['total_runs'].sum().reset_index()
overs['cumulative_runs'] = overs.groupby(['match_id', 'inning'])['total_runs'].cumsum()
overs['run_rate'] = overs['cumulative_runs'] / overs['over']
plt.figure(figsize=(12, 6))
sns.lineplot(data=overs, x='over', y='run_rate', hue='inning', estimator='mean', errorbar=None)
plt.title('Run Rate Progression by Innings')
plt.xlabel('Over')
plt.ylabel('Average Run Rate')
plt.legend(title='Inning')
plt.show()

## 3. Powerplay score vs final total correlation

In [ ]:
pp = deliveries[deliveries['over'] <= 6].groupby(['match_id', 'inning'])['total_runs'].sum().reset_index()
pp.rename(columns={'total_runs': 'powerplay_score'}, inplace=True)
final = deliveries.groupby(['match_id', 'inning'])['total_runs'].sum().reset_index()
final.rename(columns={'total_runs': 'final_total'}, inplace=True)
merged = pd.merge(pp, final, on=['match_id', 'inning'])
plt.figure(figsize=(10, 6))
sns.scatterplot(data=merged, x='powerplay_score', y='final_total', alpha=0.5)
plt.title('Powerplay Score vs Final Total')
plt.xlabel('Powerplay Score (Overs 1-6)')
plt.ylabel('Final Total')
plt.show()

## 4. Wickets fallen per over across all matches

In [ ]:
wickets = deliveries[deliveries['player_dismissed'].notna() & (deliveries['player_dismissed'] != '')]
wickets_by_over = wickets.groupby('over')['player_dismissed'].count().reset_index()
plt.figure(figsize=(10, 6))
sns.barplot(data=wickets_by_over, x='over', y='player_dismissed', color='salmon')
plt.title('Wickets Fallen per Over Across All Matches')
plt.xlabel('Over')
plt.ylabel('Total Wickets')
plt.show()

## 5. Chi-square hypothesis test (Toss Decision)

In [ ]:
print("Null Hypothesis (H0): Winning the toss does NOT significantly affect the match outcome.")
matches['toss_winner_won'] = np.where(matches['toss_winner'] == matches['winner'], 'Yes', 'No')
contingency_table = pd.crosstab(matches['toss_decision'], matches['toss_winner_won'])
chi2, p_value, dof, expected = chi2_contingency(contingency_table)
print(f"Chi-square statistic: {chi2:.4f}")
print(f"P-value: {p_value:.4f}")
if p_value < 0.05:
    print("Conclusion: Reject H0. Winning the toss significantly affects the match outcome.")
else:
    print("Conclusion: Fail to reject H0. Winning the toss does not significantly affect the match outcome.")

## 6. Top 10 batsmen by strike rate (min 200 balls faced)

In [ ]:
batsman_stats = deliveries.groupby('batsman').agg(runs=('batsman_runs', 'sum'), balls=('ball', 'count')).reset_index()
batsman_stats = batsman_stats[batsman_stats['balls'] >= 200]
batsman_stats['strike_rate'] = (batsman_stats['runs'] / batsman_stats['balls']) * 100
top_10_sr = batsman_stats.sort_values(by='strike_rate', ascending=False).head(10)
plt.figure(figsize=(12, 6))
sns.barplot(data=top_10_sr, x='strike_rate', y='batsman', hue='batsman', palette='viridis', legend=False)
plt.title('Top 10 Batsmen by Strike Rate (Min 200 balls)')
plt.xlabel('Strike Rate')
plt.ylabel('Batsman')
plt.show()

## 7. Heatmap of team vs team win/loss record

In [ ]:
matches_valid = matches[matches['result'] != 'no result'].copy()
teams = sorted(list(set(matches_valid['team1'].unique()) | set(matches_valid['team2'].unique())))
heatmap_data = pd.DataFrame(np.nan, index=teams, columns=teams)
for t1 in teams:
    for t2 in teams:
        if t1 == t2:
            continue
        subset = matches_valid[((matches_valid['team1']==t1) & (matches_valid['team2']==t2)) | ((matches_valid['team1']==t2) & (matches_valid['team2']==t1))]
        if len(subset) > 0:
            wins = len(subset[subset['winner'] == t1])
            heatmap_data.loc[t1, t2] = wins / len(subset)
plt.figure(figsize=(12, 10))
sns.heatmap(heatmap_data, annot=False, cmap='coolwarm', cbar_kws={'label': 'Win Percentage for Row Team'})
plt.title('Team vs Team Win/Loss Record (Win %)')
plt.show()